In [ ]:
#%cd ..

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

from src.ensemble_validation import get_lgbm_oof

SEED = 42

In [3]:
train = pd.read_csv("data/train.csv")

X = train.drop(columns="Survived")
y = train["Survived"]

In [4]:
best_lgbm_params = {
    "learning_rate": 0.022,
    "max_depth": 7,
    "num_leaves": 44,
    "min_child_samples": 21,
    "subsample": 1,
    "colsample_bytree": 0.707,
    "reg_alpha": 0.245,
    "reg_lambda": 0.002,
    "n_estimators": 500,
    "subsample_freq": 1,
}

oof_lgbm = get_lgbm_oof(
    X=X,
    y=y, 
    model_params=best_lgbm_params, 
    n_splits=5, 
    seed=SEED
)

oof_lgbm.shape


(891,)

In [5]:
from sklearn.metrics import accuracy_score

oof_lgbm_pred = (oof_lgbm >= 0.5).astype(int)

lgbm_oof_accuracy = accuracy_score(
    y,
    oof_lgbm_pred,
)

print("LightGBM OOF accuracy:", lgbm_oof_accuracy)

LightGBM OOF accuracy: 0.8484848484848485


In [6]:
from src.ensemble_validation import get_catboost_oof

In [7]:
best_catboost_params = {
    "depth": 6,
    "learning_rate": 0.105,
    "l2_leaf_reg": 2.183,
    "random_strength": 2.571,
}

oof_catboost = get_catboost_oof(
    X=X,
    y=y,
    model_params=best_catboost_params,
    cat_features=['Sex', 'Embarked', 'Pclass'],
    n_splits=5,
    seed=SEED,
)

catboost_oof_pred = (oof_catboost >= 0.5).astype(int)

catboost_oof_accuracy  = accuracy_score(
    y,
    catboost_oof_pred
)

print(
    "CatBoost OOF accuracy:",
    catboost_oof_accuracy
)

CatBoost OOF accuracy: 0.8327721661054994


In [8]:
from src.ensemble_validation import get_dnn_oof

In [9]:
oof_dnn = get_dnn_oof(
    X=X,
    y=y,
    n_splits=5,
    epochs=100,
    batch_size=32,
    lr=3e-3,
    seed=SEED
)

dnn_oof_pred = (oof_dnn >= 0.5).astype(int)

dnn_oof_accuracy = accuracy_score(
    y, 
    dnn_oof_pred
)

print("DNN OOF accuracy:", dnn_oof_accuracy)

DNN OOF accuracy: 0.835016835016835


In [ ]:
# Averaging

oof_average = (oof_lgbm + oof_catboost + oof_dnn) / 3

average_pred = (oof_average >= 0.5).astype(int)

average_accuracy = accuracy_score(y, average_pred)


print('avg acc:', average_accuracy)

avg acc: 0.8473625140291807


In [11]:
# Hard Voting

pred_lgbm = (oof_lgbm >= 0.5).astype(int)
pred_catboost = (oof_catboost >= 0.5).astype(int)
pred_dnn = (oof_dnn >= 0.5).astype(int)

votes = (pred_lgbm+ pred_catboost+ pred_dnn)

voting_pred = (votes >= 2).astype(int)

voting_accuracy = accuracy_score(y, voting_pred)

print("Hard Voting OOF accuracy:", voting_accuracy)

Hard Voting OOF accuracy: 0.8473625140291807


In [12]:
# stacking

X_meta = np.column_stack([
    oof_lgbm,
    oof_catboost,
    oof_dnn
])

from sklearn.linear_model import LinearRegression

meta_linear = LinearRegression()

meta_linear.fit(X_meta, y)

meta_linear_probs = meta_linear.predict(X_meta)

meta_linear_pred = (meta_linear_probs >= 0.5).astype(int)

linear_stacking_accuracy = accuracy_score(y, meta_linear_pred)

print("Linear Regression Stacking accuracy:", linear_stacking_accuracy)

Linear Regression Stacking accuracy: 0.8451178451178452


In [13]:
print(meta_linear.coef_)
print(meta_linear.intercept_)

[0.62510311 0.18449259 0.12331224]
0.030298542019619623


In [14]:
from sklearn.linear_model import Ridge

meta_ridge = Ridge(alpha=1.0)

meta_ridge.fit(X_meta, y)

ridge_scores = meta_ridge.predict(X_meta)

ridge_pred = (ridge_scores >= 0.5).astype(int)

ridge_stacking_accuracy = accuracy_score(y, ridge_pred)

print("Ridge Stacking accuracy:", ridge_stacking_accuracy)

Ridge Stacking accuracy: 0.8439955106621774


In [15]:
print("Coefficients:", meta_ridge.coef_)
print("Intercept:", meta_ridge.intercept_)


Coefficients: [0.58276297 0.20459194 0.1397602 ]
Intercept: 0.03295912005756557


In [16]:
X_meta = np.column_stack([
    oof_lgbm,
    oof_catboost,
    oof_dnn,
])

def evaluate_meta_model(
    model,
    X_meta,
    y,
    n_splits=5,
    seed=42,
):
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    scores = []

    for train_idx, val_idx in cv.split(X_meta, y):
        X_train = X_meta[train_idx]
        X_val = X_meta[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_train, y_train)

        predictions = model.predict(X_val)
        predictions = (predictions >= 0.5).astype(int)

        accuracy = accuracy_score(
            y_val,
            predictions,
        )

        scores.append(accuracy)

    scores = np.array(scores)

    return {
        "scores": scores,
        "mean": scores.mean(),
        "std": scores.std(),
    }

In [17]:
linear_result = evaluate_meta_model(
    LinearRegression(),
    X_meta,
    y,
    n_splits=5,
    seed=SEED,
)

print("Linear Stacking")
print("Scores:", linear_result["scores"])
print("Mean:", linear_result["mean"])
print("Std:", linear_result["std"])

Linear Stacking
Scores: [0.87150838 0.85955056 0.80337079 0.83707865 0.85955056]
Mean: 0.8462117883372041
Std: 0.02414344184674977


In [18]:
ridge_result = evaluate_meta_model(
    Ridge(alpha=1.0),
    X_meta,
    y,
    n_splits=5,
    seed=SEED,
)

print("Ridge Stacking")
print("Scores:", ridge_result["scores"])
print("Mean:", ridge_result["mean"])
print("Std:", ridge_result["std"])

Ridge Stacking
Scores: [0.86592179 0.85955056 0.80898876 0.83707865 0.85955056]
Mean: 0.8462180654070679
Std: 0.02103947411735518


In [19]:
final_results = pd.DataFrame({
    "Model": [
        "LightGBM + FamilySize",
        "CatBoost",
        "DNN + FamilySize + FareLog",
        "Averaging",
        "Hard Voting",
        "Linear Stacking",
        "Ridge Stacking",
    ],
    "Accuracy": [
        0.84846,
        0.83277,
        0.83500,
        0.84736,
        0.84736,
        0.84621,
        0.84622,
    ],
}).sort_values(
    "Accuracy",
    ascending=False
)

final_results

,Model,Accuracy
0,LightGBM + FamilySize,0.84846
4,Hard Voting,0.84736
3,Averaging,0.84736
6,Ridge Stacking,0.84622
5,Linear Stacking,0.84621
2,DNN + FamilySize + FareLog,0.83500
1,CatBoost,0.83277
